# LangGraph Intake Triage Notes Workflow

## 1. Project Overview

**Task:** Build a **LangGraph state graph** that converts free-text patient intake narratives into structured triage notes with clarifying questions — strictly for **administrative documentation** support.

---

### ⚠️ HEALTHCARE LIMITATIONS — READ BEFORE PROCEEDING

```
  ╔══════════════════════════════════════════════════════════════════╗
  ║  THIS NOTEBOOK IS FOR EDUCATIONAL PURPOSES ONLY.               ║
  ║                                                                ║
  ║  • NOT a diagnostic tool                                       ║
  ║  • NOT a substitute for clinical judgment                      ║
  ║  • NOT medical advice                                          ║
  ║  • NOT validated for clinical use                               ║
  ║  • NOT FDA-cleared or CE-marked                                ║
  ║                                                                ║
  ║  PURPOSE: Demonstrate LangGraph state-machine patterns using   ║
  ║  synthetic (fake) intake narratives. The pipeline produces      ║
  ║  draft administrative notes that MUST be reviewed and           ║
  ║  validated by a licensed healthcare professional before any     ║
  ║  clinical use.                                                 ║
  ║                                                                ║
  ║  REAL-WORLD USE: Any production healthcare NLP system must     ║
  ║  comply with HIPAA, local medical device regulations, and      ║
  ║  institutional review board (IRB) requirements. A local LLM    ║
  ║  (Ollama) is intentionally used here to avoid sending data     ║
  ║  to external APIs, but this alone does NOT constitute HIPAA    ║
  ║  compliance.                                                   ║
  ╚══════════════════════════════════════════════════════════════════╝
```

---

**Pipeline (administrative documentation only):**

```
  START
    │
    │  STATE: {intake_text, ...empty fields...}
    ▼
  ┌─────────────────────────┐
  │  parse_intake            │  extract structured fields from free text
  └──────────┬──────────────┘
             │  STATE += {parsed_fields}
             ▼
  ┌─────────────────────────┐
  │  assess_urgency          │  classify administrative priority level
  └──────────┬──────────────┘
             │  STATE += {urgency_assessment}
             ▼
  ┌─────────────────────────┐
  │  generate_questions      │  identify missing info, produce questions
  └──────────┬──────────────┘
             │  STATE += {clarifying_questions}
             ▼
  ┌─────────────────────────┐
  │  compile_triage_note     │  assemble structured note from all fields
  └──────────┬──────────────┘
             │  STATE += {triage_note}
             ▼
  ┌─────────────────────────┐
  │  review_completeness     │  check note quality, route
  └──────────┬──────────────┘
        conditional
      ┌─────┴─────┐
      ▼           ▼
   complete    incomplete
      │           │
      │           └→ generate_questions (loop, capped)
      ▼
  ┌─────────────────────────┐
  │  finalize_output         │  add disclaimers, format final output
  └──────────┬──────────────┘
             ▼
           END
```

**Stack:** `LangGraph`, `ChatOllama` + `qwen3.5:9b` (local — no data leaves the machine), `pandas`

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Design a **TypedDict state schema** for structured extraction |
| 2 | Build **extraction nodes** that parse free text into typed fields |
| 3 | Implement a **completeness review loop** with conditional routing |
| 4 | Generate **clarifying questions** for missing or ambiguous information |
| 5 | Embed **domain-specific safety constraints** throughout the pipeline |
| 6 | Understand **partial state updates** — each node owns specific fields |
| 7 | Evaluate across multiple synthetic intake scenarios |

## 3. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langgraph pandas numpy

print("Dependencies: langchain, langgraph, pandas")
print("LLM: Ollama with qwen3.5:9b (local — no external API calls)")

## 4. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict
from datetime import datetime

import pandas as pd

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"

print("All imports OK")
print(f"LLM: {LLM_MODEL} (local Ollama — no data sent externally)")

## 5. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


DISCLAIMER = (
    "[DISCLAIMER] This is an AI-generated DRAFT for administrative documentation "
    "purposes only. It is NOT a clinical assessment, NOT medical advice, and MUST "
    "be reviewed and validated by a licensed healthcare professional before any use."
)

# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Synthetic Intake Data & State Schema

## 6. Synthetic Intake Narratives

These are **entirely fictional** intake texts. No real patient information is used. The scenarios are designed to demonstrate different extraction challenges.

> **All names, dates, and clinical details are fabricated for educational purposes.**

In [ ]:
SYNTHETIC_INTAKES = [
    {
        "id": "SYNTH-001",
        "label": "Structured narrative — most fields present",
        "text": (
            "Jane Doe, 42-year-old female, presents to the clinic on 2024-11-15 "
            "with a 3-day history of worsening lower back pain radiating to the "
            "right leg. She rates the pain 7/10. No fever. She takes ibuprofen "
            "400mg twice daily with partial relief. Past medical history includes "
            "a lumbar disc herniation (L4-L5) diagnosed in 2021. She has no known "
            "drug allergies. She works as a warehouse associate and has been unable "
            "to work for the past two days. She is requesting a referral to "
            "physical therapy."
        ),
    },
    {
        "id": "SYNTH-002",
        "label": "Sparse narrative — many fields missing",
        "text": (
            "Mr. Rodriguez called in about chest tightness that comes and goes. "
            "Has been happening for about a week. He didn't say much else. "
            "Sounded concerned."
        ),
    },
    {
        "id": "SYNTH-003",
        "label": "Pediatric with caregiver — third-party report",
        "text": (
            "Mother calling about her 6-year-old son Tyler Brooks. He's had a "
            "persistent cough for five days with a low-grade fever of about "
            "100.2F yesterday. He's been eating less but still drinking fluids. "
            "She tried children's acetaminophen. No known allergies. He's up to "
            "date on vaccinations. They don't have a regular pediatrician and "
            "she'd like to get him seen today if possible. She also mentioned "
            "he has mild asthma and uses an inhaler occasionally."
        ),
    },
    {
        "id": "SYNTH-004",
        "label": "Mental health — sensitive topic with vague language",
        "text": (
            "A 28-year-old named Sam Kim called saying they've been feeling "
            "'really off' for the past month. Trouble sleeping, no appetite, "
            "can't focus at work. They mentioned being 'overwhelmed' and asked "
            "if the clinic has counseling services. No medications currently. "
            "They said they don't feel like hurting themselves but feel 'stuck'."
        ),
    },
]

print(f"Synthetic intakes: {len(SYNTHETIC_INTAKES)}")
for s in SYNTHETIC_INTAKES:
    print(f"  [{s['id']}] {s['label']} ({len(s['text'])} chars)")

## 7. State Schema

| Field | Written By | Purpose |
|-------|-----------|--------|
| `intake_text` | Input | Raw free-text narrative (immutable) |
| `intake_id` | Input | Tracking identifier |
| `parsed_fields` | parse_intake | Extracted structured data as JSON string |
| `urgency_assessment` | assess_urgency | Priority classification with rationale |
| `clarifying_questions` | generate_questions | Questions about missing/ambiguous info |
| `triage_note` | compile_triage_note | Structured administrative note |
| `completeness_score` | review_completeness | 0-100 completeness rating |
| `completeness_gaps` | review_completeness | What's still missing |
| `review_decision` | review_completeness | complete / incomplete |
| `revision_count` | review_completeness | Loop counter |
| `final_output` | finalize_output | Disclaimer-wrapped final document |
| `current_node` | all nodes | Debugging |

In [ ]:
class TriageState(TypedDict):
    """State schema for intake triage note generation.

    ALL outputs are DRAFT administrative notes — NOT clinical assessments.
    """
    intake_text: str
    intake_id: str
    parsed_fields: str
    urgency_assessment: str
    clarifying_questions: str
    triage_note: str
    completeness_score: int
    completeness_gaps: str
    review_decision: str
    revision_count: int
    final_output: str
    current_node: str


print("State schema: TriageState")
for name, typ in TriageState.__annotations__.items():
    print(f"  {name}: {typ}")

---

# Part B — Build the Graph Nodes

## 8. Healthcare Safety System Prompt

Every LLM call in this pipeline includes a safety constraint prefix. This is a **defense-in-depth** measure — even if a single node's prompt is modified, the system-level constraint persists.

In [ ]:
SAFETY_PREFIX = """CRITICAL CONSTRAINTS — You MUST follow these in every response:
1. You are an ADMINISTRATIVE DOCUMENTATION ASSISTANT, not a clinician.
2. You MUST NOT diagnose, prescribe, or recommend treatments.
3. You MUST NOT suggest clinical severity levels or triage categories
   (e.g., ESI levels, CTAS scores) — those require licensed clinical judgment.
4. Every output must include: "DRAFT — Requires licensed professional review."
5. If the intake describes an emergency (e.g., chest pain, difficulty breathing,
   suicidal ideation), you MUST prepend: "FLAG: This intake may describe an
   urgent situation. Please ensure immediate clinical review."
6. Do NOT interpret symptoms — only extract and organize what was stated.
7. Use the phrase "patient reports" or "caller states" — never "patient has"
   or "patient is experiencing" (those imply clinical confirmation).
"""

print("Safety prefix loaded (applied to every LLM call)")
print(f"Length: {len(SAFETY_PREFIX)} chars")
print()
print("Key constraints:")
print("  • No diagnoses, no prescriptions, no treatment recs")
print("  • No clinical severity or triage category assignment")
print("  • Emergency flag for potentially urgent intakes")
print("  • Language guard: 'patient reports' not 'patient has'")

## 9. Node 1: Parse Intake

Extracts structured fields from free text. This is **extraction only** — the node does not interpret, assess, or diagnose.

```
  Transition: START → parse_intake
  Precondition:  intake_text is non-empty
  Postcondition: parsed_fields contains JSON with extracted fields
```

In [ ]:
PARSE_SYSTEM = SAFETY_PREFIX + """
Task: Extract structured fields from an intake narrative.
Return ONLY valid JSON with these fields (use null for missing):
{
  "name": "...",
  "age": "...",
  "sex_or_gender": "...",
  "contact_type": "in-person | phone | other",
  "reporter": "self | caregiver | other",
  "reported_symptoms": ["symptom 1", "symptom 2"],
  "reported_duration": "...",
  "reported_severity": "...",
  "reported_medications": ["med 1"],
  "reported_allergies": "...",
  "reported_medical_history": ["item 1"],
  "reported_request": "what the patient/caller is asking for",
  "missing_fields": ["fields with no info in the text"]
}

RULES:
- Extract ONLY what is explicitly stated. Do NOT infer.
- Use "reported_" prefix — these are unverified statements.
- If a field has no information, set it to null and add to missing_fields."""


def parse_intake(state: TriageState) -> dict:
    """Node: Extract structured fields from free-text intake.

    Transition: START → parse_intake
    Precondition: intake_text exists
    Postcondition: parsed_fields populated with JSON
    """
    print("  [NODE] parse_intake")

    raw = ask(
        f"Extract structured fields from this intake:\n\n{state['intake_text']}",
        system=PARSE_SYSTEM,
        temperature=0.1,
    )

    parsed = parse_json(raw)
    if parsed:
        fields_str = json.dumps(parsed, indent=2)
        missing = parsed.get("missing_fields", [])
        print(f"    Extracted {len([v for v in parsed.values() if v is not None])} fields")
        print(f"    Missing: {missing if missing else 'none identified'}")
    else:
        fields_str = raw
        print("    Warning: could not parse JSON — stored raw text")

    return {"parsed_fields": fields_str, "current_node": "parse_intake"}


print("Node defined: parse_intake")
print("  READS:  intake_text")
print("  WRITES: parsed_fields")
print("  SAFETY: extraction only, 'reported_' prefix, no inference")

## 10. Node 2: Assess Urgency

Classifies **administrative priority** — NOT clinical triage level. This determines documentation scheduling, not clinical care.

```
  Transition: parse_intake → assess_urgency
  Precondition:  parsed_fields exists with extracted data
  Postcondition: urgency_assessment with priority and rationale
```

> **Why NOT clinical triage?** Assigning ESI levels, CTAS scores, or clinical urgency requires a licensed professional examining the patient. An LLM working from a secondhand text narrative cannot and should not do this.

In [ ]:
URGENCY_SYSTEM = SAFETY_PREFIX + """
Task: Classify the ADMINISTRATIVE PRIORITY of this intake for documentation
scheduling purposes. This is NOT a clinical triage level.

Administrative priority levels (for scheduling documentation review only):
  - FLAG_FOR_IMMEDIATE_REVIEW: intake describes potentially urgent symptoms
    that should be reviewed by clinical staff as soon as possible
  - PRIORITY: intake has time-sensitive elements (short quote validity,
    explicit same-day request, worsening trajectory)
  - ROUTINE: no time-sensitive elements identified

Return valid JSON:
{
  "admin_priority": "FLAG_FOR_IMMEDIATE_REVIEW | PRIORITY | ROUTINE",
  "rationale": "why this priority was assigned",
  "urgent_flag": true/false,
  "urgent_flag_reason": "if flagged, why immediate review is recommended"
}

RULES:
- You are assigning DOCUMENTATION SCHEDULING priority, not clinical severity.
- If the intake mentions chest pain, difficulty breathing, suicidal ideation,
  severe bleeding, or loss of consciousness → FLAG_FOR_IMMEDIATE_REVIEW.
- Do NOT assign ESI, CTAS, or any clinical triage score."""


def assess_urgency(state: TriageState) -> dict:
    """Node: Classify administrative documentation priority.

    Transition: parse_intake → assess_urgency
    Precondition: parsed_fields exists
    Postcondition: urgency_assessment populated

    NOTE: This is NOT clinical triage — it is documentation scheduling.
    """
    print("  [NODE] assess_urgency")

    raw = ask(
        f"PARSED INTAKE DATA:\n{state['parsed_fields']}\n\n"
        f"ORIGINAL TEXT:\n{state['intake_text']}\n\n"
        f"Classify the administrative priority.",
        system=URGENCY_SYSTEM,
        temperature=0.1,
    )

    result = parse_json(raw)
    if result:
        priority = result.get("admin_priority", "ROUTINE")
        flagged = result.get("urgent_flag", False)
        assessment = json.dumps(result, indent=2)
        print(f"    Priority: {priority}")
        if flagged:
            print(f"    ⚠ FLAGGED: {result.get('urgent_flag_reason', 'see report')}")
    else:
        assessment = raw
        print("    Warning: could not parse JSON — stored raw text")

    return {"urgency_assessment": assessment, "current_node": "assess_urgency"}


print("Node defined: assess_urgency")
print("  READS:  parsed_fields, intake_text")
print("  WRITES: urgency_assessment")
print("  SAFETY: administrative priority only, not clinical triage")

## 11. Node 3: Generate Clarifying Questions

Produces questions that a front-desk coordinator would ask to complete the intake record. These are **administrative data-gathering questions**, not clinical history questions.

```
  Transition: assess_urgency → generate_questions
  Precondition:  parsed_fields and urgency_assessment both exist
  Postcondition: clarifying_questions with categorized questions
```

In [ ]:
QUESTIONS_SYSTEM = SAFETY_PREFIX + """
Task: Generate clarifying questions an intake coordinator would ask to
complete the administrative record. These are data-gathering questions,
NOT clinical history questions.

Categories:
  - DEMOGRAPHICS: name, DOB, contact info, insurance
  - SYMPTOM DETAILS: onset, duration, severity, aggravating/relieving factors
  - MEDICATION & ALLERGIES: current meds, known allergies, pharmacy
  - HISTORY: past conditions, surgeries, family history mentioned
  - LOGISTICS: preferred appointment time, transportation needs, interpreter

Return valid JSON:
{
  "questions": [
    {"category": "...", "question": "...", "reason": "why this info is needed"},
    ...
  ],
  "total_questions": N,
  "priority_question": "the single most important question to ask first"
}

RULES:
- Only ask about information NOT already provided in the parsed fields.
- DO NOT ask diagnostic questions (e.g., 'do you think it could be X?').
- Keep questions simple and appropriate for a non-clinical coordinator."""


def generate_questions(state: TriageState) -> dict:
    """Node: Generate clarifying questions for the intake coordinator.

    Transition: assess_urgency → generate_questions (or review loop)
    Precondition: parsed_fields, urgency_assessment exist
    Postcondition: clarifying_questions populated
    """
    print("  [NODE] generate_questions")

    revision_note = ""
    if state.get("revision_count", 0) > 0 and state.get("completeness_gaps"):
        revision_note = (
            f"\nPREVIOUS REVIEW found these gaps — focus new questions here:\n"
            f"{state['completeness_gaps']}\n\n"
            f"PREVIOUS QUESTIONS (do not repeat):\n"
            f"{state['clarifying_questions'][:500]}\n"
        )
        print(f"    Revision round {state['revision_count']} — targeting gaps")

    raw = ask(
        f"PARSED FIELDS:\n{state['parsed_fields']}\n\n"
        f"URGENCY ASSESSMENT:\n{state['urgency_assessment']}\n"
        f"{revision_note}\n"
        f"Generate clarifying questions for the intake coordinator.",
        system=QUESTIONS_SYSTEM,
        temperature=0.2,
    )

    result = parse_json(raw)
    if result:
        questions = result.get("questions", [])
        output = json.dumps(result, indent=2)
        print(f"    Generated {len(questions)} questions")
        if result.get("priority_question"):
            print(f"    Priority: {result['priority_question'][:80]}")
    else:
        output = raw
        print("    Warning: could not parse JSON — stored raw text")

    return {"clarifying_questions": output, "current_node": "generate_questions"}


print("Node defined: generate_questions")
print("  READS:  parsed_fields, urgency_assessment, completeness_gaps (on revision)")
print("  WRITES: clarifying_questions")
print("  SAFETY: administrative data-gathering only, no diagnostic questions")

## 12. Node 4: Compile Triage Note

Assembles all extracted and generated data into a structured administrative note.

```
  Transition: generate_questions → compile_triage_note
  Precondition:  parsed_fields, urgency_assessment, clarifying_questions exist
  Postcondition: triage_note with structured administrative note
```

In [ ]:
NOTE_SYSTEM = SAFETY_PREFIX + """
Task: Compile a structured administrative triage note from parsed intake data.
This note is a DRAFT for clinical staff to review — it is not a clinical record.

Format the note with these sections:
1. HEADER: intake ID, date, admin priority
2. REPORTED DEMOGRAPHICS: name, age, sex/gender
3. REPORTED CHIEF CONCERN: what the patient/caller described (use 'reports' language)
4. REPORTED SYMPTOM DETAILS: duration, severity, associated factors
5. REPORTED MEDICATIONS & ALLERGIES
6. REPORTED HISTORY
7. PATIENT/CALLER REQUEST: what they are asking for
8. ADMINISTRATIVE NOTES: priority level, documentation gaps
9. CLARIFYING QUESTIONS: for coordinator follow-up
10. DISCLAIMER

RULES:
- Use 'caller reports' / 'patient reports' — NEVER 'patient has' or 'patient presents with'.
- For missing fields: write '[NOT PROVIDED — see clarifying questions]'.
- End with the disclaimer: 'DRAFT — Requires licensed professional review.'"""


def compile_triage_note(state: TriageState) -> dict:
    """Node: Compile structured administrative triage note.

    Transition: generate_questions → compile_triage_note
    Precondition: parsed_fields, urgency_assessment, clarifying_questions exist
    Postcondition: triage_note populated
    """
    print("  [NODE] compile_triage_note")

    note = ask(
        f"INTAKE ID: {state['intake_id']}\n"
        f"DATE: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"
        f"PARSED FIELDS:\n{state['parsed_fields']}\n\n"
        f"URGENCY ASSESSMENT:\n{state['urgency_assessment']}\n\n"
        f"CLARIFYING QUESTIONS:\n{state['clarifying_questions'][:800]}\n\n"
        f"Compile the structured administrative triage note.",
        system=NOTE_SYSTEM,
        temperature=0.2,
    )

    # Ensure disclaimer is present
    if "DRAFT" not in note and "disclaimer" not in note.lower():
        note += "\n\nDRAFT — Requires licensed professional review."

    print(f"    Note: {len(note)} chars")
    return {"triage_note": note, "current_node": "compile_triage_note"}


print("Node defined: compile_triage_note")
print("  READS:  parsed_fields, urgency_assessment, clarifying_questions, intake_id")
print("  WRITES: triage_note")
print("  SAFETY: 'reports' language enforced, disclaimer appended if missing")

## 13. Node 5: Review Completeness

Checks whether the triage note covers minimum required fields. Routes to "complete" or "incomplete" (revision loop).

```
  Transition: compile_triage_note → review_completeness
  Precondition:  triage_note exists
  Postcondition: completeness_score, completeness_gaps, review_decision populated

  Routing:
    complete   (score ≥ 70 OR max revisions hit) → finalize_output
    incomplete (score < 70 AND under cap)        → generate_questions
```

In [ ]:
REVIEW_SYSTEM = SAFETY_PREFIX + """
Task: Review the triage note for DOCUMENTATION COMPLETENESS.
This is an administrative quality check, not a clinical review.

Check these mandatory fields (10 points each, 100 max):
  1. Patient/caller name
  2. Age or date of birth
  3. Reported chief concern
  4. Reported duration/onset
  5. Reported severity (if applicable)
  6. Reported medications
  7. Reported allergies
  8. Reported history
  9. Patient/caller request
  10. Disclaimer present

Return valid JSON:
{
  "completeness_score": 0-100,
  "fields_present": ["name", "age", ...],
  "fields_missing": ["allergies", ...],
  "gaps_description": "what is missing and why it matters",
  "decision": "complete" or "incomplete"
}"""


def review_completeness(state: TriageState) -> dict:
    """Node: Check triage note completeness and route.

    Transition: compile_triage_note → review_completeness
    Precondition: triage_note exists
    Postcondition: score, gaps, decision populated
    """
    print("  [NODE] review_completeness")

    raw = ask(
        f"TRIAGE NOTE:\n{state['triage_note'][:2000]}\n\n"
        f"Review for documentation completeness.",
        system=REVIEW_SYSTEM,
        temperature=0.1,
    )

    result = parse_json(raw)
    revisions = state.get("revision_count", 0)

    if result:
        score = result.get("completeness_score", 50)
        gaps = result.get("gaps_description", "")
        decision = "complete" if score >= 70 else "incomplete"
    else:
        score = 60
        gaps = "Could not parse review — defaulting to incomplete"
        decision = "incomplete"

    print(f"    Score: {score}/100, Decision: {decision}")
    if gaps:
        print(f"    Gaps: {gaps[:120]}")

    return {
        "completeness_score": score,
        "completeness_gaps": gaps,
        "review_decision": decision,
        "revision_count": revisions + 1,
        "current_node": "review_completeness",
    }


print("Node defined: review_completeness")
print("  READS:  triage_note")
print("  WRITES: completeness_score, completeness_gaps, review_decision, revision_count")

## 14. Node 6: Finalize Output

Wraps the final output with mandatory disclaimers and formatting.

```
  Transition: review_completeness → finalize_output (terminal)
  Precondition:  ALL prior fields populated, review_decision is complete (or at cap)
  Postcondition: final_output with disclaimer-wrapped document
```

In [ ]:
def finalize_output(state: TriageState) -> dict:
    """Node: Add disclaimers and produce the final output.

    Transition: review_completeness → finalize_output
    Precondition: triage_note, clarifying_questions, urgency_assessment exist
    Postcondition: final_output ready
    """
    print("  [NODE] finalize_output")

    # Parse urgency for the header
    urgency_data = parse_json(state["urgency_assessment"])
    priority = urgency_data.get("admin_priority", "UNKNOWN") if urgency_data else "UNKNOWN"
    is_flagged = urgency_data.get("urgent_flag", False) if urgency_data else False

    header_lines = [
        "=" * 70,
        DISCLAIMER,
        "=" * 70,
    ]

    if is_flagged:
        flag_reason = urgency_data.get("urgent_flag_reason", "Potentially urgent intake") if urgency_data else ""
        header_lines.extend([
            "",
            "⚠" * 35,
            f"FLAG: This intake may describe an urgent situation.",
            f"Reason: {flag_reason}",
            "Please ensure immediate clinical review.",
            "⚠" * 35,
            "",
        ])

    output = (
        "\n".join(header_lines) + "\n\n"
        f"INTAKE ID:           {state['intake_id']}\n"
        f"ADMIN PRIORITY:      {priority}\n"
        f"COMPLETENESS SCORE:  {state.get('completeness_score', 'N/A')}/100\n"
        f"REVIEW ROUNDS:       {state.get('revision_count', 0)}\n"
        f"\n{'─' * 70}\n"
        f"STRUCTURED TRIAGE NOTE\n"
        f"{'─' * 70}\n\n"
        f"{state['triage_note']}\n"
        f"\n{'─' * 70}\n"
        f"CLARIFYING QUESTIONS FOR COORDINATOR\n"
        f"{'─' * 70}\n\n"
        f"{state['clarifying_questions']}\n"
        f"\n{'═' * 70}\n"
        f"{DISCLAIMER}\n"
        f"{'═' * 70}\n"
    )

    print(f"    Final output: {len(output)} chars")
    print(f"    Flagged: {is_flagged}")
    return {"final_output": output, "current_node": "finalize_output"}


print("Node defined: finalize_output")
print("  READS:  triage_note, clarifying_questions, urgency_assessment, intake_id")
print("  WRITES: final_output")
print("  SAFETY: disclaimer at top AND bottom of output")

---

# Part C — Routing & Graph Assembly

## 15. Routing After Review

```
  ┌──────────────────────────────────────────────────────────────┐
  │  ROUTING LOGIC                                              │
  ├──────────────────────────────────────────────────────────────┤
  │                                                              │
  │  IF review_decision == "complete":                          │
  │    → finalize_output (note is adequate)                     │
  │                                                              │
  │  IF review_decision == "incomplete" AND under revision cap: │
  │    → generate_questions (loop back for more targeted Qs)    │
  │                                                              │
  │  IF at revision cap:                                         │
  │    → finalize_output (best-effort with gaps documented)     │
  └──────────────────────────────────────────────────────────────┘
```

In [ ]:
MAX_REVISIONS = 2


def route_after_review(state: TriageState) -> str:
    """Conditional edge: route based on note completeness."""
    decision = state.get("review_decision", "complete")
    revisions = state.get("revision_count", 0)

    if decision == "complete":
        print(f"    [ROUTE] complete (score ≥ 70) → finalize_output")
        return "finalize_output"

    if revisions >= MAX_REVISIONS:
        print(f"    [ROUTE] incomplete at cap ({MAX_REVISIONS}) → finalize_output")
        return "finalize_output"

    print(f"    [ROUTE] incomplete (round {revisions}) → generate_questions")
    return "generate_questions"


print(f"Routing: route_after_review (max {MAX_REVISIONS} revision rounds)")
print("  complete              → finalize_output")
print("  incomplete (under cap) → generate_questions")
print("  incomplete (at cap)   → finalize_output")

## 16. Assemble the StateGraph

In [ ]:
workflow = StateGraph(TriageState)

# Register nodes
workflow.add_node("parse_intake", parse_intake)
workflow.add_node("assess_urgency", assess_urgency)
workflow.add_node("generate_questions", generate_questions)
workflow.add_node("compile_triage_note", compile_triage_note)
workflow.add_node("review_completeness", review_completeness)
workflow.add_node("finalize_output", finalize_output)

# T1: START → parse_intake
workflow.add_edge(START, "parse_intake")

# T2: parse_intake → assess_urgency
workflow.add_edge("parse_intake", "assess_urgency")

# T3: assess_urgency → generate_questions
workflow.add_edge("assess_urgency", "generate_questions")

# T4: generate_questions → compile_triage_note
workflow.add_edge("generate_questions", "compile_triage_note")

# T5: compile_triage_note → review_completeness
workflow.add_edge("compile_triage_note", "review_completeness")

# T6: conditional — route based on completeness
workflow.add_conditional_edges(
    "review_completeness",
    route_after_review,
    {
        "finalize_output": "finalize_output",
        "generate_questions": "generate_questions",
    },
)

# T7: finalize_output → END
workflow.add_edge("finalize_output", END)

# Compile
app = workflow.compile()

print("Graph compiled!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print()
print("State transitions:")
print("  T1: START            → parse_intake         (extract fields)")
print("  T2: parse_intake     → assess_urgency       (admin priority)")
print("  T3: assess_urgency   → generate_questions   (missing info Qs)")
print("  T4: generate_questions→ compile_triage_note (assemble note)")
print("  T5: compile_note     → review_completeness  (quality check)")
print("  T6: review           → finalize_output      (complete)")
print("      review           → generate_questions   (incomplete loop)")
print("  T7: finalize         → END")

In [ ]:
# Visualize graph
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering not available: {e}")
    print("\nGraph (text):")
    print("  __start__ --> parse_intake --> assess_urgency --> generate_questions")
    print("  generate_questions --> compile_triage_note --> review_completeness")
    print("  review_completeness --(complete)--> finalize_output --> __end__")
    print("  review_completeness --(incomplete)--> generate_questions")

---

# Part D — Execute the Pipeline

## 17. Run on Intake 1 — Structured Narrative

In [ ]:
def make_initial_state(intake: dict) -> TriageState:
    return {
        "intake_text": intake["text"],
        "intake_id": intake["id"],
        "parsed_fields": "",
        "urgency_assessment": "",
        "clarifying_questions": "",
        "triage_note": "",
        "completeness_score": 0,
        "completeness_gaps": "",
        "review_decision": "",
        "revision_count": 0,
        "final_output": "",
        "current_node": "start",
    }


intake_1 = SYNTHETIC_INTAKES[0]
print(f"Pipeline: [{intake_1['id']}] {intake_1['label']}")
print(f"Text: {intake_1['text'][:100]}...")
print("=" * 70)

result_1 = app.invoke(
    make_initial_state(intake_1),
    {"recursion_limit": 25},
)

print(f"\nComplete.")
print(f"  Completeness: {result_1['completeness_score']}/100")
print(f"  Revisions: {result_1['revision_count']}")
print(f"  Output: {len(result_1['final_output'])} chars")

## 18. View the Structured Triage Note

In [ ]:
print(f"TRIAGE NOTE — {intake_1['id']}")
print("=" * 70)
wrap_print(result_1["triage_note"])

## 19. View Clarifying Questions

In [ ]:
print(f"CLARIFYING QUESTIONS — {intake_1['id']}")
print("=" * 70)

q_data = parse_json(result_1["clarifying_questions"])
if q_data and "questions" in q_data:
    for i, q in enumerate(q_data["questions"], 1):
        cat = q.get("category", "GENERAL")
        question = q.get("question", "")
        reason = q.get("reason", "")
        print(f"  Q{i} [{cat}]: {question}")
        if reason:
            print(f"      Why: {reason}")
    if q_data.get("priority_question"):
        print(f"\n  ★ Ask first: {q_data['priority_question']}")
else:
    wrap_print(result_1["clarifying_questions"])

## 20. Streaming — Watch State Transitions

In [ ]:
print("STREAMING EXECUTION — Intake 1")
print("=" * 65)

step = 0
for chunk in app.stream(make_initial_state(intake_1), {"recursion_limit": 25}):
    step += 1
    for node_name, node_output in chunk.items():
        keys = [k for k in node_output if k != "current_node"]
        details = []
        for k in keys:
            v = node_output[k]
            if isinstance(v, str) and len(v) > 50:
                details.append(f"{k}: {len(v)} chars")
            elif isinstance(v, int):
                details.append(f"{k}: {v}")
            else:
                details.append(f"{k}: {v}")
        print(f"  Step {step}: {node_name:<24} → {', '.join(details)}")

print(f"\nTotal steps: {step}")

---

# Part E — Run All Synthetic Intakes

## 21. Process All Four Scenarios

In [ ]:
print("PROCESSING ALL SYNTHETIC INTAKES")
print("=" * 70)

all_results = []

for i, intake in enumerate(SYNTHETIC_INTAKES, 1):
    print(f"\n{'=' * 70}")
    print(f"INTAKE {i}/{len(SYNTHETIC_INTAKES)}: [{intake['id']}] {intake['label']}")
    print("=" * 70)

    result = app.invoke(
        make_initial_state(intake),
        {"recursion_limit": 25},
    )
    all_results.append(result)

    urgency = parse_json(result["urgency_assessment"])
    priority = urgency.get("admin_priority", "?") if urgency else "?"
    flagged = urgency.get("urgent_flag", False) if urgency else False

    print(f"  → Score: {result['completeness_score']}/100 | "
          f"Priority: {priority} | "
          f"Flagged: {flagged} | "
          f"Revisions: {result['revision_count']}")

print(f"\nAll {len(SYNTHETIC_INTAKES)} intakes processed.")

## 22. Comparison Dashboard

In [ ]:
print("INTAKE COMPARISON DASHBOARD")
print("=" * 95)

rows = []
for intake, result in zip(SYNTHETIC_INTAKES, all_results):
    urgency = parse_json(result["urgency_assessment"])
    priority = urgency.get("admin_priority", "?") if urgency else "?"
    flagged = urgency.get("urgent_flag", False) if urgency else False

    questions = parse_json(result["clarifying_questions"])
    n_questions = len(questions.get("questions", [])) if questions else 0

    rows.append({
        "ID": intake["id"],
        "Scenario": intake["label"][:30],
        "Priority": priority,
        "Flagged": "⚠ YES" if flagged else "No",
        "Score": result["completeness_score"],
        "Questions": n_questions,
        "Revisions": result["revision_count"],
        "Note Size": len(result["triage_note"]),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f"\n  Average completeness: {df['Score'].mean():.0f}/100")
print(f"  Average questions: {df['Questions'].mean():.1f}")
print(f"  Flagged intakes: {sum(1 for r in rows if '⚠' in r['Flagged'])}")

## 23. View Final Output — Sparse Narrative (Intake 2)

This is the most challenging scenario — the intake has very little information. The pipeline should flag many gaps and generate focused clarifying questions.

In [ ]:
print("FINAL OUTPUT — SYNTH-002 (sparse narrative)")
print("=" * 70)
wrap_print(all_results[1]["final_output"][:2000])
if len(all_results[1]["final_output"]) > 2000:
    print(f"  ... [{len(all_results[1]['final_output']) - 2000} more chars]")

## 24. View Final Output — Mental Health Scenario (Intake 4)

This scenario tests whether the pipeline handles sensitive topics appropriately — flagging urgency without overstepping into clinical judgment.

In [ ]:
print("FINAL OUTPUT — SYNTH-004 (mental health intake)")
print("=" * 70)
wrap_print(all_results[3]["final_output"][:2000])
if len(all_results[3]["final_output"]) > 2000:
    print(f"  ... [{len(all_results[3]['final_output']) - 2000} more chars]")

---

# Part F — Analysis & Review

## 25. State Accumulation Per Intake

In [ ]:
print("STATE SIZE ANALYSIS")
print("=" * 70)

for intake, result in zip(SYNTHETIC_INTAKES, all_results):
    fields = [
        ("parsed_fields", result["parsed_fields"]),
        ("urgency_assessment", result["urgency_assessment"]),
        ("clarifying_questions", result["clarifying_questions"]),
        ("triage_note", result["triage_note"]),
        ("final_output", result["final_output"]),
    ]

    total = sum(len(v) for _, v in fields)
    print(f"\n  [{intake['id']}] {intake['label'][:35]}")
    for name, content in fields:
        bar = "█" * min(len(content) // 40, 30)
        print(f"    {name:<24} {len(content):>5} ch  {bar}")
    print(f"    {'TOTAL':<24} {total:>5} ch")

## 26. Healthcare Safety Verification

Let's verify that the pipeline's safety constraints were respected across all runs.

In [ ]:
print("SAFETY CONSTRAINT VERIFICATION")
print("=" * 70)

unsafe_phrases = [
    "patient has", "patient is experiencing", "diagnosed with",
    "prescribe", "prescription", "you should take",
    "ESI level", "ESI 1", "ESI 2", "ESI 3", "ESI 4", "ESI 5",
    "CTAS", "triage category", "clinical severity",
]

required_phrases = ["DRAFT", "review"]

for intake, result in zip(SYNTHETIC_INTAKES, all_results):
    note = result["triage_note"].lower()
    output = result["final_output"].lower()

    print(f"\n  [{intake['id']}] {intake['label'][:35]}")

    # Check for unsafe phrases
    found_unsafe = []
    for phrase in unsafe_phrases:
        if phrase.lower() in note or phrase.lower() in output:
            found_unsafe.append(phrase)

    if found_unsafe:
        print(f"    ⚠ UNSAFE phrases found: {found_unsafe}")
    else:
        print(f"    ✓ No unsafe phrases detected")

    # Check for required phrases
    for phrase in required_phrases:
        if phrase.lower() in output:
            print(f"    ✓ Required '{phrase}' present")
        else:
            print(f"    ⚠ Required '{phrase}' MISSING")

    # Check disclaimer
    if "disclaimer" in output or "draft" in output:
        print(f"    ✓ Disclaimer present")
    else:
        print(f"    ⚠ Disclaimer may be missing")

print("\n" + "=" * 70)
print("Note: LLM outputs are non-deterministic. Some phrases may occasionally")
print("appear despite prompting. Production systems need additional safeguards")
print("(regex filters, blocked-phrase lists, human review).")

## 27. Exercises

### Exercise 1: Add an Allergy Interaction Check
Add a `check_interactions` node between `parse_intake` and `assess_urgency` that compares reported medications against reported allergies and flags potential concerns. Remember: flag for review only — do NOT make clinical assessments.

### Exercise 2: Multi-Language Support
Modify `parse_intake` to detect the intake language and translate to English before structured extraction. Add a `detected_language` field to the state schema.

### Exercise 3: Intake Deduplication
Add a node that checks whether a new intake text is similar to a previously processed one (using embedding similarity). If similarity > 0.85, flag it as a potential duplicate. Store previous intakes in a list.

### Mini Challenge: Coordinator Workflow
Replace the automatic `generate_questions` → `compile_triage_note` transition with a simulated coordinator interaction: present the questions, accept simulated answers via a dictionary, merge the answers into `parsed_fields`, then recompile the note.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Intakes processed:     {len(all_results)}")
print(f"Average completeness:  {df['Score'].mean():.0f}/100")
print(f"Average questions:     {df['Questions'].mean():.1f}")
print(f"Flagged for review:    {sum(1 for r in rows if '⚠' in r['Flagged'])}")
print()
print("State transitions:")
print("  T1: START      → parse_intake        + parsed_fields")
print("  T2: parse      → assess_urgency      + urgency_assessment")
print("  T3: urgency    → generate_questions   + clarifying_questions")
print("  T4: questions  → compile_triage_note  + triage_note")
print("  T5: compile    → review_completeness  + score, gaps, decision")
print("  T6: review     → finalize_output      (complete)")
print("      review     → generate_questions   (incomplete loop)")
print("  T7: finalize   → END                  + final_output")
print()
print("Nodes built:")
print("  1. parse_intake         — extracts structured fields from text")
print("  2. assess_urgency       — admin priority (NOT clinical triage)")
print("  3. generate_questions   — clarifying Qs for coordinator")
print("  4. compile_triage_note  — assembles structured admin note")
print("  5. review_completeness  — quality check with revision routing")
print("  6. finalize_output      — disclaimer-wrapped final document")
print()
print("Healthcare safety constraints:")
print("  • SAFETY_PREFIX on every LLM call")
print("  • 'reports' language — never 'has' or 'presents with'")
print("  • No diagnosis, prescription, or treatment recommendations")
print("  • No clinical triage scores (ESI, CTAS)")
print("  • Urgent intake flagging (for clinical staff review)")
print("  • Disclaimer at top and bottom of every output")
print("  • Local LLM only — no data sent to external APIs")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Healthcare LLM pipelines need defense-in-depth** — system prompt, node-level, and output-level constraints |
| 2 | **Use "reported" language** — the LLM extracts what was stated, not what is clinically confirmed |
| 3 | **Administrative priority ≠ clinical triage** — assigning ESI/CTAS levels requires licensed professionals |
| 4 | **Flag, don't diagnose** — when the intake sounds urgent, flag it for immediate human review |
| 5 | **Completeness loops add value** — sparse intakes get targeted follow-up questions |
| 6 | **Cap all loops** — MAX_REVISIONS prevents infinite cycling |
| 7 | **Local LLMs reduce data exposure** — but do NOT constitute HIPAA compliance alone |
| 8 | **Disclaimers must be structural** — enforced at the code level, not just in prompts |
| 9 | **Safety checks need post-hoc verification** — scan outputs for forbidden phrases |
| 10 | **Production systems need more** — regex filters, blocked-phrase lists, audit logs, IRB approval |

---

### ⚠️ Final Reminder

This notebook demonstrates **NLP pipeline engineering patterns** using synthetic data. It is **not** a medical device, clinical decision support system, or substitute for licensed healthcare professionals. Any adaptation for real-world healthcare use must comply with applicable regulations (HIPAA, FDA, MDR, local medical device laws) and receive appropriate institutional review.

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*